# Readout mock (tProc v2)

This notebook replays recorded qubit readout traces through a DAC-to-ADC loopback, so that the NN classifier on the board (`l2_w400_ternary_h4_s100`) scores real readout signals instead of a constant pulse. It then compares the on-board accuracy with the NN's offline accuracy.

The traces are the QICK ZCU216 20240528 dataset that the NN was trained on: decimated readout traces (307.2 MHz, 770 samples, 14-bit I/Q), labelled ground or excited. Each trace is resampled to the DAC rate (6881.28 Msps) and played as an arbitrary envelope. The readout captures it and the NN scores trace samples 100-499, as it would for a real qubit. The NN classifies the samples as they arrive, so the replay has to match the recording: same phase, same sample positions, and the same scale, or 1/g of it with the NN's input gain set to g (its `scaling_factor` register: 1, 2, 4 or 8). The loopback cannot reach the dataset's amplitude with an arbitrary envelope, so the replay runs at 1/g and the NN multiplies it back. Most of this notebook calibrates that.

It is the tProc v2 counterpart of the tProc v1 notebooks in `qick_dev/qick_ml/readout_mock/` (`readout_mock_malab*.ipynb`). Those replayed older ZCU111 traces to an earlier NN with two logits. It reuses the loopback setup of [`../send_receive_pulse.ipynb`](../send_receive_pulse.ipynb); run that one first if the bench is new.

The notebook runs in this order:

1. Check the repo, import this repo's `qick_lib` and load the NN bitstream.
2. Load the traces: 1000 shots of the test set copied to the board (default), or the 20 HLS testbench shots (see `README.md`).
3. Calibrate the replay: amplitude, timing, phase, I/Q orientation, then a check and a correction on real traces.
4. Replay the shots, read one NN logit per shot and score them.

## Bench setup

As in [`../send_receive_pulse.ipynb`](../send_receive_pulse.ipynb#Bench-setup): an SMA cable from the generator's DAC (`0_230` on JHC3 for `DAC = 230`, `0_228` on JHC1 for `DAC = 228`) to ADC `0_226` on JHC7. Nothing else should be on the path: the calibration below measures the loopback's gain and phase, and they are assumed to stay put during the run.

## Environment

In [ ]:
import subprocess

!git branch
!echo -n "git rev:  "; git rev-parse --short HEAD
!echo -n "git date: "; git log -1 --format=%cd

branch_name = subprocess.getoutput("git rev-parse --abbrev-ref HEAD").replace('/', '-')
print('Bitstream directory: ../216/{}'.format(branch_name))

In [ ]:
# Use this repo's own qick_lib (it matches this branch's firmware), as
# ../send_receive_pulse.ipynb does; readout_mock_lib puts ../ (qick_ml_lib) on the path
import sys, os
repo_qick_lib = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..', 'qick_lib'))
if repo_qick_lib not in sys.path:
    sys.path.insert(0, repo_qick_lib)

from qick import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import qick
print('Using qick from:', qick.__file__)

# readout_mock_lib first: it puts ../ on sys.path, where qick_ml_lib is
import readout_mock_lib as rm
import qick_ml_lib
from qick_ml_lib import (calibrate_phase, reset_classifier, configure_classifier,
                          get_classifier_prediction_count, print_classifier_buffer)

## Bitstream

An NN build from `../216/<branch>/` (see [`../send_receive_pulse.ipynb`](../send_receive_pulse.ipynb#Bitstream) for the builds and how they get there).

In [ ]:
# Generator DAC, to match the cabling: 228 (0_228, JHC1) or 230 (0_230, JHC3)
DAC = 230
# 'nn' or 'nn_ila' (the NN with ILAs on it)
BUILD = 'nn'

CUSTOM_BIT = '../216/{}/qick_216_tprocv2_dac{}_{}.bit'.format(branch_name, DAC, BUILD)
CUSTOM_BIT_FULL_PATH = os.path.normpath(os.path.join(os.getcwd(), CUSTOM_BIT))
print('Custom bitstream:', CUSTOM_BIT_FULL_PATH)
if not os.path.isfile(os.path.splitext(CUSTOM_BIT_FULL_PATH)[0] + '.hwh'):
    raise FileNotFoundError('Missing HWH file next to the bitstream')

soc = QickSoc(bitfile=CUSTOM_BIT_FULL_PATH)
soccfg = soc
print(soccfg)

if not (hasattr(soccfg, 'NN_0') and hasattr(soccfg, 'axi_blk_bram_ctrl_0')):
    raise RuntimeError('This bitstream has no NN')
qick_ml_lib.set_soccfg(soccfg)

# The NN listens to readout 0
GEN_CH = 0
RO_CH = 0
print('Generator channel {}: {}'.format(GEN_CH, soccfg._describe_dac(soccfg['gens'][GEN_CH]['dac'])))
print('Readout channel   {}: {}'.format(RO_CH, soccfg._describe_adc(soccfg['readouts'][RO_CH]['adc'])))
F_DAC, SAMPS_PER_CLK, MAXV = rm.gen_params(soccfg, GEN_CH)
print('Generator: {:.2f} Msps, {} samples per clock, envelope full scale {:.0f}'.format(F_DAC, SAMPS_PER_CLK, MAXV))

### Memory

The board has 3.9 GB of RAM and 1 GB of swap on the SD card, and it stops responding when both fill up (it needed a power cycle on 2026-09-30). `rm.memory_report()` prints the available memory, the swap in use and this kernel's size; it warns below 500 MB. The notebook calls it again after loading the traces and after the NN run. Every open kernel that loaded a bitstream keeps its memory: shut down the ones you do not need (Kernel > Shut Down in the other notebooks).

In [ ]:
mem = rm.memory_report('after loading the bitstream')

## Dataset

By default the notebook replays `N_SHOTS = 1000` shots of the test set, 500 ground and 500 excited picked at random (`SEED`), from the copy on the board in `DATA_DIR` (`X_NAME`, an exact int16 copy of the full traces; see `README.md` for how to make it). The test set has 100,000 shots with labels, so the run gives an accuracy, which the Results section compares with the C model's. `N_SHOTS = None` replays all of them (about 5.5 hours at 0.2 s per shot).

`DATA_DIR = None` loads instead the 20 test shots of the NN's HLS testbench from `../ip/20240528/two_layers_w400_ternary_h4_s100_20240528_hls4ml_prj.zip` (the first and the last 10 shots of the test set), with nothing to copy. They come with no labels, only samples 100-499, and the exact logits of the C simulation, which is what the NN IP computes from those inputs. So they check the replay shot by shot: a faithful replay gives the same logits.

In [ ]:
DATA_DIR = '/home/xilinx/data/20240528/000_770'   # the test set on the board; None: the 20 HLS testbench shots
X_NAME = 'X_test_000_770_int16.npy'   # int16 copy of X_test_000_770.npy (see README.md)
N_SHOTS = 1000           # balanced random subset (half per class); None: all shots
SEED = 0

if DATA_DIR is None:
    dataset = rm.load_dataset(None)
else:
    dataset = rm.load_dataset(DATA_DIR, x_name=X_NAME)
idx = rm.select_shots(dataset, N_SHOTS, SEED)
print('{}: replaying {} shots'.format(dataset.name, len(idx)))

win = np.stack([dataset.window(i) for i in idx])
peak = np.hypot(win[..., 0], win[..., 1]).max()
print('Samples 100-499: mean |IQ| {:.0f}, peak |IQ| {:.0f} ADC units'.format(
    np.hypot(win[..., 0], win[..., 1]).mean(), peak))

plt.figure(figsize=(10, 4))
plt.plot(np.arange(100, 500), dataset.window(idx[0])[:, 0], label='I')
plt.plot(np.arange(100, 500), dataset.window(idx[0])[:, 1], label='Q')
plt.xlabel('Trace sample')
plt.ylabel('ADC units')
plt.title('Shot {} (samples 100-499, the NN window)'.format(idx[0]))
plt.legend()
plt.grid(True)
plt.show()

mem = rm.memory_report('after loading the traces')

## Loopback configuration

The same carrier and start timing as [`../send_receive_pulse.ipynb`](../send_receive_pulse.ipynb#Loopback-pulse-configuration). The trace is up- and down-converted digitally, so the carrier frequency does not show up in the capture; it only sets the loopback's gain and phase. Each replay covers the NN window plus `MARGIN` samples on each side, tapered to zero.

In [ ]:
config = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 250.0,              # MHz, carrier
    'phase': 0,                 # degrees, set by the phase calibration below
    'trig_time': 62 / 430.08,   # microseconds after the pulse start, the start value for the timing calibration
    'ro_len': 2.5,              # microseconds (768 samples)
}
MARGIN = 20                     # readout samples replayed on each side of the window

## Amplitude

The replay has to reach the ADC at the dataset's scale divided by the NN's input gain `NN_GAIN`. This replays rectangles at a few fractions of full scale and extrapolates the readout magnitude at full scale (`FULL_SCALE`, in ADC units). A replay at 1/`NN_GAIN` needs `FULL_SCALE` at or above the dataset's peak |IQ| / `NN_GAIN`; below it the loudest samples are clipped. `NN_GAIN = None` picks the smallest gain (1, 2, 4 or 8) that fits; an arbitrary envelope reaches only about a third of the dataset's peak on the ZCU216 loopback, so this is usually 4. A gain of g costs log2(g) bits of the replay and multiplies the loopback's own noise by g (the C model's accuracy does not change with the input rounded to multiples of 8). If even 8 falls short, lower the ADC attenuation (printed below; `soc.rf.set_adc_attenuator(adc, dB)`) or try another carrier frequency, and run this cell again.

In [ ]:
ALLOW_CLIPPING = False
NN_GAIN = None           # None: the smallest of 1, 2, 4, 8 that fits the loopback

adc = soccfg['readouts'][RO_CH]['adc']
if hasattr(soc.rf, 'get_adc_attenuator'):
    print('ADC {} attenuator: {} dB'.format(adc, soc.rf.get_adc_attenuator(adc)))
else:
    print('ADC {}: this qick version has no attenuator control'.format(adc))

FULL_SCALE, resid = rm.measure_full_scale(soc, config)
print('Full scale {:.0f} ADC units, dataset peak {:.0f} ({:.0%} of full scale)'.format(FULL_SCALE, peak, peak / FULL_SCALE))
if resid > 0.02:
    print('WARNING: the loopback gain is not linear ({:.1%} residual)'.format(resid))
if NN_GAIN is None:
    NN_GAIN = rm.nn_gain_for(peak, FULL_SCALE) or 8
print('NN input gain {}: replay at 1/{} of the dataset scale, peak {:.0f} ADC units ({:.0%} of full scale)'.format(
    NN_GAIN, NN_GAIN, peak / NN_GAIN, peak / NN_GAIN / FULL_SCALE))
if peak / NN_GAIN > FULL_SCALE:
    msg = 'The loopback reaches only {:.0f} ADC units, the replay needs {:.0f}: it would clip'.format(FULL_SCALE, peak / NN_GAIN)
    if not ALLOW_CLIPPING:
        raise RuntimeError(msg + ' (set ALLOW_CLIPPING = True to go on anyway)')
    print('WARNING: ' + msg)

# A rectangle at the replay's mean level, for the calibrations below
CAL_AMP = np.hypot(win[..., 0], win[..., 1]).mean() / NN_GAIN / FULL_SCALE * MAXV

## Timing

Dataset sample k has to land on trace sample k, so that the NN sees samples 100-499 of the recording. This replays a rectangle laid out like a trace envelope and moves it until its rising edge is on sample 100. It moves the readout trigger in whole tProc timing cycles (0.714 samples) and then delays the envelope by leading zero DAC samples (1/22.4 sample each).

In [ ]:
config['trig_time'], LEAD, edge = rm.align_timing(soc, config, CAL_AMP, MARGIN)

iq = rm.acquire_trace(soc, dict(config, env=rm.rect_envelope(soccfg, GEN_CH, CAL_AMP, margin=MARGIN, lead=LEAD)))
plt.figure(figsize=(10, 4))
plt.plot(np.hypot(iq[:, 0], iq[:, 1]), label='|IQ|')
plt.axvline(100, ls='--', color='red')
plt.axvline(500, ls='--', color='red', label='NN window')
plt.xlabel('Trace sample')
plt.ylabel('ADC units')
plt.title('Timing: rising edge at {:.2f}'.format(edge))
plt.legend()
plt.grid(True)
plt.show()

## Phase and I/Q orientation

The loopback rotates the I/Q plane by a phase that changes on every bitstream load, and it may mirror it. `calibrate_phase` (from `qick_ml_lib`) turns the carrier phase so that an I-only envelope arrives on +I. A Q-only envelope then arrives on +Q (kept) or on -Q (mirrored: the envelope's Q is negated). This replaces the v1 mock's hand-set I/Q swap and `DAC_phase = -90`.

In [ ]:
rect = rm.rect_envelope(soccfg, GEN_CH, CAL_AMP, margin=MARGIN, lead=LEAD)
config['phase'] = calibrate_phase(soc, dict(config, env=rect), target_deg=0.0, prog_cls=rm.ArbPulseProgram)
CONJ = rm.check_iq_orientation(soc, config, CAL_AMP)
print('I/Q orientation:', 'mirrored (Q negated)' if CONJ else 'kept')

cal = {'scale': MAXV / FULL_SCALE, 'conj': CONJ, 'lead': LEAD, 'nn_gain': NN_GAIN}

## Replay check

This replays a few traces and compares each captured window with the recording (divided by `NN_GAIN`, the scale the replay aims at). A least-squares fit `captured = g * data` gives the gain and angle left over (1 and 0 deg when the calibration holds), plus the sub-sample lag and the RMS difference. The rectangles above only probe the carrier. The traces carry noise up to ~140 MHz, where the loopback's response is not flat, so the mean `g` and lag of these replays go into the calibration as a trim, over a few passes. The traces are broadband, so the timing has to hold to a small fraction of a sample: 0.1 sample is already ~0.2 rad at 100 MHz. The RMS difference that remains is what the DAC and ADC filters change; it does not go away.

On the 20 testbench shots (2026-09-30) the rectangle calibration alone gave board logits within 1% of the C simulation (slope 0.998, correlation 0.9997), while the fit on 5 noise-dominated traces was biased (|g| ~0.8, the filters attenuating the traces' high-frequency noise) and unstable (angle +-25 deg, lag +-0.6 samples between passes), so folding it in made the logits worse. `N_PASSES = 0` keeps the rectangle calibration and only reports the fit; with passes, `refine_calibration` folds in the angle and the lag, not the magnitude (`trim_magnitude=False`).

In [ ]:
N_REFINE = 5    # shots per pass
N_PASSES = 0    # 0: keep the rectangle calibration (see the note above)
for _ in range(N_PASSES):
    cal, errs = rm.refine_calibration(soc, dataset, idx[:N_REFINE], cal, config, MARGIN)
print(cal)

In [ ]:
# Overlay one replay on its recording
i = idx[0]
env, clipped = rm.trace_to_envelope(dataset, i, cal, soccfg, GEN_CH, MARGIN)
iq = rm.acquire_trace(soc, dict(config, env=env))
cap = iq[100:500]
rec = dataset.window(i) / NN_GAIN   # the recording at the replay's scale
e = rm.replay_error(rec, cap)
print('Shot {}: gain {gain:.3f}, angle {angle_deg:+.2f} deg, lag {lag:+.2f}, rms {rms:.0f} ADC units '
      '({rms_rel:.1%}), corr I {corr_i:.4f}, Q {corr_q:.4f}; {c} clipped DAC samples'.format(i, c=clipped, **e))

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
t = np.arange(100, 500)
for k, name in enumerate('IQ'):
    ax[k].plot(t, rec[:, k], label='recorded / {}'.format(NN_GAIN), color='gray')
    ax[k].plot(t, cap[:, k], label='replayed', lw=1)
    ax[k].set_ylabel(name + ' (ADC units)')
    ax[k].legend(loc='upper right')
    ax[k].grid(True)
ax[2].plot(t, cap - rec)
ax[2].set_ylabel('replayed - recorded / {}'.format(NN_GAIN))
ax[2].set_xlabel('Trace sample')
ax[2].grid(True)
plt.show()

## NN run

This resets the classifier (the calibration shots above also triggered it), configures its window on samples 100-499 (`WINDOW_OFFSET = 95`, as in [`../send_receive_pulse.ipynb`](../send_receive_pulse.ipynb#NN-classifier-setup)) and its input gain to `NN_GAIN`, and replays the shots, one program each. Every shot prints a character: `.` when the prediction matches the label (or, for the testbench shots, the sign of the expected logit), `*` when it does not.

In [ ]:
WINDOW_SIZE = 400
WINDOW_OFFSET = 95
SCALING_FACTOR = NN_GAIN   # the NN's input gain: it multiplies the 1/NN_GAIN replay back

reset_classifier()
configure_classifier(WINDOW_SIZE, WINDOW_OFFSET, SCALING_FACTOR, debug=True)

In [ ]:
%%time
res = rm.run_shots(soc, dataset, idx, cal, config, MARGIN)
print('NN predictions: {} (expected {}), {:.2f} s per shot, {} shots with clipped samples'.format(
    res['count'], len(idx), res['seconds'] / len(idx), np.count_nonzero(res['clipped'])))
mem = rm.memory_report('after the run')

### Results

In [ ]:
s = rm.score(dataset, res)
if 'accuracy' in s:
    print('Shots: {}'.format(s['n']))
    print('               board     C model (100,000 shots)')
    print('accuracy     {:7.3%}   {:7.3%}'.format(s['accuracy'], rm.C_MODEL_ACCURACY))
    print('fidelity     {:7.3%}   {:7.3%}'.format(s['fidelity'], rm.C_MODEL_FIDELITY))
    print('ground       {:7.3%}'.format(s['accuracy_ground']))
    print('excited      {:7.3%}'.format(s['accuracy_excited']))
if dataset.ref_logits is not None:
    ref = dataset.ref_logits[idx]
    print('Sign agreement with the C simulation: {:.0%}, logit correlation {:.4f}'.format(
        s['sign_agreement'], s['logit_corr']))
    print('shot   expected      board   difference')
    for k, i in enumerate(idx):
        print('{:4d} {:10.0f} {:10.0f} {:+10.0f} {}'.format(
            i, ref[k], res['logits'][k], res['logits'][k] - ref[k],
            '' if (ref[k] >= 0) == (res['logits'][k] >= 0) else '<<< sign'))

In [ ]:
plt.figure(figsize=(10, 4))
if dataset.y is not None:
    y = dataset.y[idx]
    bins = np.linspace(res['logits'].min(), res['logits'].max(), 60)
    plt.hist(res['logits'][y == 0], bins, alpha=0.5, label='ground')
    plt.hist(res['logits'][y == 1], bins, alpha=0.5, label='excited')
    plt.axvline(0, color='k', ls='--')
    plt.xlabel('Board logit')
    plt.ylabel('Shots')
else:
    ref = dataset.ref_logits[idx]
    plt.plot(ref, res['logits'], 'o')
    lim = [min(ref.min(), res['logits'].min()), max(ref.max(), res['logits'].max())]
    plt.plot(lim, lim, 'k--', label='board = expected')
    plt.axhline(0, color='gray', lw=0.5)
    plt.axvline(0, color='gray', lw=0.5)
    plt.xlabel('Expected logit (C simulation)')
    plt.ylabel('Board logit')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Save the run: shots, logits, captured windows and the calibration
import time
os.makedirs('results', exist_ok=True)
out = 'results/readout_mock_{}.npz'.format(time.strftime('%Y%m%d_%H%M%S'))
np.savez_compressed(out, dataset=dataset.name, bitfile=CUSTOM_BIT_FULL_PATH, config=repr(config),
                    cal=repr(cal), full_scale=FULL_SCALE, score=repr(s), **res)
print('Saved', out)

In [ ]:
print_classifier_buffer(index_lo=0, index_hi=7)